# Part 1 - Multimodal RAG over a read-only knowledge base
Pipeline: PyMuPDF -> text chunks + figure crops -> Qwen2.5-VL descriptions -> ChromaDB + BM25 -> hybrid (RRF) retrieval -> grounded JSON answers with per-conversation memory.
Data: attach the Kaggle competition data (Document*.pdf, kb_manifest.json, Questions.csv) or set `DATA_DIR`.

In [1]:
# Cell 1 - run information (required)
import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

{'commit': '678da737', 'python': '3.12.10', 'torch': '2.11.0+cu128', 'gpu': 'NVIDIA GeForce RTX 5090', 'seed': 266}


In [2]:
import os, sys, json, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd
pd.set_option("display.max_colwidth", 120)

## 1.1 Ingestion & indexing

In [3]:
from p1_rag.pipeline import build, answer_file
from p1_rag.index import Retriever
from p1_rag.rag import RAGSystem
t0 = time.time()
final, items, figures = build(work_dir="work_p1", mode="hybrid")      # reranker_id="BAAI/bge-reranker-base" for the bonus
print("ingestion+index seconds:", round(time.time() - t0))
fdf = pd.DataFrame(figures)
print(len(items), "indexed items;", len(figures), "figures/tables;", int(fdf.untrusted.sum()), "inside Appendix C (excluded from retrieval)")
fdf[["doc_id", "page", "label", "kind", "trust_level", "untrusted", "cropped", "injection_in_image"]]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


ingestion+index seconds: 454
302 indexed items; 36 figures/tables; 1 inside Appendix C (excluded from retrieval)


,doc_id,page,label,kind,trust_level,untrusted,cropped,injection_in_image
0,DOC1,1,Table 1-1,table,high,0,True,False
1,DOC1,2,Figure 1-1,figure,high,0,False,False
2,DOC1,3,Figure 1-2,figure,high,0,False,False
3,DOC1,4,Figure 1-3,figure,high,0,False,False
4,DOC1,5,Table 1-2,table,high,0,True,False
5,DOC1,5,Figure 1-4,figure,high,0,False,False
6,DOC1,7,Table 1-3,table,high,0,True,False
7,DOC1,8,Figure 1-5,figure,high,0,True,False
8,DOC1,10,Figure 1-6,figure,high,0,False,False
9,DOC1,11,Table 1-4,table,high,0,True,False


## 1.2 / 1.3 Retrieval, answering, conversations - sanity check

In [4]:
chat = final.new_chat()
print(chat.answer("What does Figure 1-1 show?"))
print(final.last_retrieved[:5])
n_before = final.retriever.col.count()

W1109 16:18:08.329000 50208 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


{'answer': 'Figure 1-1 shows stock prices in the United States, Euro area, and emerging economies from 2007 to 2010.', 'status': 'answered', 'citations': ['DOC1:Figure 1-1', 'DOC1:Figure 1-6'], 'figure_references': ['Figure 1-1']}
[{'doc_id': 'DOC1', 'page': 2, 'figure_id': 'Figure 1-1'}, {'doc_id': 'DOC1', 'page': 10, 'figure_id': 'Figure 1-6'}, {'doc_id': 'DOC1', 'page': 5, 'figure_id': 'Figure 1-4'}, {'doc_id': 'DOC1', 'page': 3, 'figure_id': 'Figure 1-2'}, {'doc_id': 'DOC1', 'page': 20, 'figure_id': 'Figure 2-1'}]


## 1.4 Golden set (25 questions written by us from the PDFs, committed BEFORE tuning)
See `p1_rag/golden.py` for the schema. Save as `golden_set.json` in the repo root.

In [5]:
from p1_rag import golden
G = golden.load("golden_set.json")
print("coverage problems:", golden.check_coverage(G) or "none", "| n =", len(G))

coverage problems: none | n = 25


In [6]:
# baseline = dense retrieval only, no follow-up rewriting; final = hybrid + conversation handling
baseline = RAGSystem(Retriever(items, figures, "work_p1/chroma_dense", mode="dense", embedder=final.retriever.embedder),
                     final.vlm, conversation=False, llm=final.llm)   # same answering model; only retrieval/conversation differ
rows = {}
for name, system in [("baseline", baseline), ("final", final)]:
    recs = golden.run_golden(system, G)
    rows[name] = golden.summarize(recs)
    pd.DataFrame(recs).to_csv(f"golden_results_{name}.csv", index=False)
results = pd.DataFrame(rows).T
results

,correct_out_of_25,n,recall@5,citation_accuracy,median_sec_per_question
baseline,23.0,25.0,0.913,0.645,3.56
final,21.0,25.0,0.957,0.750,3.42


In [7]:
# Read-only knowledge base: answering must never change the vector store
assert final.retriever.col.count() == n_before, "vector store changed during answering!"
print("vector store unchanged:", n_before, "items")
golden.human_sheet(recs)   # (recs = final-system results from the loop above) both members rate 1-5 in human_eval_p1.csv, then:
# golden.human_summary("human_eval_p1.csv")

vector store unchanged: 302 items


## C4 safety analysis - untrusted Appendix C, false claims, injections

In [8]:
from p1_rag.text_utils import has_injection
untrusted = [i for i in items if i["untrusted"]]
print("Appendix C items kept out of retrieval:", len(untrusted))
print("figures whose VLM transcript contained instruction-like text:", [f["label"] for f in figures if f["injection_in_image"]])
print("text chunks (trusted) that read like injections before sanitising:", sum(has_injection(i["text"]) for i in items if not i["untrusted"]))

Appendix C items kept out of retrieval: 3
figures whose VLM transcript contained instruction-like text: ['Figure C-1']
text chunks (trusted) that read like injections before sanitising: 1


## 1.5 Kaggle submission (reads QUESTIONS_PATH, default Questions.csv; no manual steps)

In [9]:
from p1_rag.ingest import find_data_dir
QUESTIONS_PATH = os.environ.get("QUESTIONS_PATH", os.path.join(find_data_dir(), "Questions.csv"))
sub = answer_file(final, QUESTIONS_PATH, "submission.csv")
sub.head()

D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


... (26 outputs left out)

D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
D:\LAB2\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


,id,conversation_id,question_id,answer,figure_references,citations,status
0,1,C001,Q001,"The global economic crisis around 2008 was sparked by a collapse in stock prices, especially in the United States, w...",NONE,DOC1:p2,answered
1,2,C002,Q002,"Unemployment rates going up can indicate that the economy is not using its resources efficiently, with many workers ...",NONE,DOC1:p24,answered
2,3,C003,Q003,"Economists measure economic growth without price changes by calculating real GDP, which is the sum of the quantities...",NONE,DOC1:p19,answered
3,4,C004,Q004,"In 2009, the world economy experienced a decline in output by 3.5%, a dramatic increase in unemployment to nearly 10...",NONE,DOC1:p2,answered
4,5,C005,Q005,U.S. unemployment increased dramatically to nearly 10% in 2009 after the 2008 crisis.,Figure 2-3,DOC1:p5,answered


In [10]:
import subprocess
print(subprocess.run([sys.executable, os.path.join(find_data_dir(), "validate_submission.py"), "submission.csv", QUESTIONS_PATH], capture_output=True, text=True).stdout)

valid: 40 rows

